# Análisis Exploratorio de Datos — Demanda horaria de bicicletas 🚲
## Predicción de demanda para un sistema de *bike sharing* — *Washington D.C.*

**Desafío de Ciencia de Datos — Duoc UC 2026**

---

### Integrantes del equipo

| Nombre |
|--------|
| Sebastián González Pino |
| Hernán Lippke Vega |
| Matías Arauz |
| Michelangelo Bandelli |

**Asignatura:** *Desafío de Ciencia de Datos*

**Repositorio:** `github.com/SebastianRGP/desafio-movilidad-bikeshare`

---

### 1. Contexto del problema

Un operador de **bicicletas compartidas** enfrenta un problema logístico recurrente: la demanda **no es
uniforme en el tiempo ni en el espacio**. A ciertas horas las estaciones del centro quedan vacías y las
periféricas saturadas; en otras ocurre lo inverso. Cada bicicleta que falta cuando el usuario la necesita es
un viaje perdido, y cada camión de redistribución movido "a ciegas" es un costo operacional evitable.

La pregunta de negocio que guía todo el proyecto es deliberadamente concreta:

> **¿Cuántas bicicletas se usarán en la próxima hora?**

Anticipar esa cifra habilita tres decisiones operativas de valor directo:

1. **Redistribuir la flota** hacia donde habrá demanda *antes* de que falten unidades.
2. **Planificar el mantenimiento** en las horas valle, sin afectar el nivel de servicio.
3. **Dimensionar la operación** (turnos, personal, inventario) anticipando *peaks* de *commuting* y
   estacionalidad.

### 1.1 Objetivo de este notebook

Este documento corresponde a la etapa de **Análisis Exploratorio de Datos (EDA)** del proyecto. Su propósito
**no** es entrenar un modelo, sino **entender el fenómeno y justificar con evidencia** las decisiones que se
toman aguas abajo (ingeniería de características, estrategia de validación, elección de métricas). Es decir:
todo lo que el pipeline hace después, aquí queda **fundamentado**.

Concretamente, el EDA busca responder:

| # | Pregunta de análisis | Decisión que fundamenta |
|---|----------------------|-------------------------|
| P1 | ¿Cómo se distribuye la demanda (`cnt`)? ¿Es simétrica? | Métrica de error y tratamiento de outliers |
| P2 | ¿Existe un patrón horario estable? ¿Cambia entre día laboral y fin de semana? | Features cíclicas y de calendario |
| P3 | ¿Cuánto influye el **clima real** (temperatura, precipitación)? | Justificación de la 2ª fuente de datos |
| P4 | ¿Los **feriados** alteran el patrón? | Justificación de la 3ª fuente de datos |
| P5 | ¿La demanda es **persistente** (la hora anterior predice la actual)? | Features autorregresivas (*lags*) |
| P6 | ¿Hay tendencia y estacionalidad anual? | Estrategia de *split* temporal y riesgo de deriva |

### 1.2 Alcance y posición en el pipeline

```
Fuentes → [ ETL: extract → transform → features → load ] → ► ESTE NOTEBOOK (EDA) ◄ → Modelo → API → Dashboard
```

El EDA se ejecuta **sobre el dataset ya procesado** (`data/processed/bikeshare_features.parquet`), es decir,
**después** del ETL. Esta decisión es intencional: exploramos exactamente **la misma tabla que verá el
modelo**, de modo que cualquier hallazgo aquí es directamente accionable sobre las features reales y no
sobre una versión intermedia que el modelo nunca usará.

## 2. Importación de librerías y configuración estética

Cargamos las librerías de manipulación (`pandas`, `numpy`) y de visualización. Para los gráficos usamos
**Plotly** (`plotly.graph_objects` / `plotly.subplots`) como librería principal: a diferencia de una imagen
estática, cada figura queda **interactiva** — zoom, *pan*, *tooltips* con el valor exacto al pasar el mouse,
mostrar/ocultar series desde la leyenda — lo que es especialmente valioso en gráficos densos como el mapa de
calor hora × día, el correlograma de 72 barras o la matriz de correlación, donde leer un valor puntual en una
imagen estática obliga a entrecerrar los ojos. Plotly ya es una **dependencia del proyecto** (la usa el
`dashboard`), por lo que adoptarla como librería principal del EDA no añade peso adicional al contenedor
(`Dockerfile`).

Definimos además una **paleta institucional única** y una **plantilla (`template`) de Plotly** con parámetros
globales, para que todas las figuras del informe compartan identidad visual y sean legibles al proyectarlas
en la defensa.


In [ ]:
# ---- Manipulación y análisis de datos ----
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Visualización (Plotly como librería principal: gráficos interactivos) ----
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# ---- Paleta institucional del proyecto (movilidad urbana) ----
color_principal  = "#1B3A4B"   # Azul petróleo  → estructura, referencias
color_secundario = "#E4572E"   # Naranja acento → alertas, valores sobre el promedio
color_terciario  = "#4C9F70"   # Verde          → valores bajo el promedio, clima favorable
color_cuarto     = "#F3A712"   # Ámbar          → destacados secundarios
color_quinto     = "#8896AB"   # Gris azulado   → elementos neutros / de fondo
color_sexto      = "#A62639"   # Rojo profundo  → umbrales críticos
color_septimo    = "#2A9D8F"   # Turquesa       → series comparativas

paleta = [color_principal, color_secundario, color_terciario,
          color_cuarto, color_septimo, color_sexto, color_quinto]

# Escala de color continua derivada de la misma paleta (para heatmaps y degradados)
escala_bici = [
    [0.00, "#F7F4EF"], [0.25, color_terciario], [0.55, color_cuarto],
    [0.80, color_secundario], [1.00, color_sexto],
]


# Convierte un color HEX de la paleta a rgba(...) con transparencia (para bandas y rellenos)
def rgba(color_hex, alpha):
    color_hex = color_hex.lstrip("#")
    r, g, b = (int(color_hex[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


# ---- Plantilla Plotly institucional: para que todas las figuras compartan identidad visual ----
plantilla_bici = go.layout.Template(
    layout=go.Layout(
        colorway=paleta,
        font=dict(family="Arial, Helvetica, sans-serif", size=12, color=color_principal),
        title=dict(font=dict(size=16, color=color_principal), x=0.02, xanchor="left"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(gridcolor="#DDE3EA", zeroline=False, showline=True, linecolor="#DDE3EA"),
        yaxis=dict(gridcolor="#DDE3EA", zeroline=False, showline=True, linecolor="#DDE3EA"),
        legend=dict(bgcolor="rgba(255,255,255,0)", bordercolor="rgba(0,0,0,0)"),
        margin=dict(t=70, l=60, r=30, b=50),
    )
)
pio.templates["bici"] = plantilla_bici
pio.templates.default = "plotly_white+bici"


# Formatea un número con separador de miles "." (para textos e insights impresos)
def miles(v):
    return f"{v:,.0f}".replace(",", ".")


print("Librerías cargadas. Paleta y plantilla Plotly institucional definidas.")


## 3. Carga de los datos procesados

Cargamos la tabla que produce el ETL mediante la función `load_processed()` del **paquete instalable**
`bikeshare`. Usar la función del paquete —y no un `read_parquet` con ruta fija— es una decisión de
**reproducibilidad y trazabilidad**: el notebook consume exactamente el mismo artefacto que consumen la API
y el dashboard, por lo que ninguno de los tres puede quedar leyendo una versión distinta de los datos.

Se incluye un **respaldo** (`fallback`) que localiza el `.parquet` mediante `Path.rglob` si el paquete no
estuviera instalado en el entorno del evaluador, de modo que el informe sea ejecutable de forma autónoma.

In [ ]:
def cargar_datos():
    '''Carga el dataset procesado por el ETL.

    Estrategia en dos niveles:
      1) Via el paquete `bikeshare` (ruta oficial, la misma que usan API y dashboard).
      2) Fallback: busqueda recursiva del parquet, para que el notebook corra en cualquier entorno.
    '''
    try:
        from bikeshare.etl.load import load_processed
        datos = load_processed()
        print("Origen: paquete `bikeshare` → load_processed()")
        return datos
    except Exception as e:  # entorno sin el paquete instalado
        print(f"[aviso] No se pudo usar el paquete ({type(e).__name__}). Buscando el parquet en disco...")
        for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
            candidatos = sorted(base.rglob("bikeshare_features.parquet"))
            if candidatos:
                print(f"Origen: archivo local → {candidatos[0]}")
                return pd.read_parquet(candidatos[0])
        raise FileNotFoundError(
            "No se encontró `bikeshare_features.parquet`. Ejecuta primero el ETL: `python -m bikeshare.etl.run`"
        )


df = cargar_datos()
print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas".replace(",", "."))

In [ ]:
# Vista transpuesta de las primeras filas: con ~40 columnas, la transposición (.T) permite
# revisar todas las variables de una sola pasada, sin scroll horizontal.
df.head(3).T

### 3.1 Las tres fuentes integradas

El dataset no proviene de un único archivo: es el resultado de un **ETL multi-fuente** que une, por
`datetime`, tres orígenes de naturaleza distinta. Integrar fuentes es un requisito del desafío, pero aquí
además tiene una **justificación analítica**: el registro histórico de viajes dice *cuánto* se pedaleó, pero
no *por qué*; el clima y el calendario aportan ese contexto.

| # | Fuente | Tipo de acceso | Qué aporta | Por qué la incluimos |
|---|--------|----------------|------------|----------------------|
| 1 | **UCI Bike Sharing Dataset** (Washington D.C., 2011–2012, horario) | Archivo estático (CSV) | Variable objetivo `cnt` y contexto base (estación, clima categórico) | Es el registro real de la operación: sin él no hay objetivo que predecir |
| 2 | **Open-Meteo API** (*Historical Weather*) | API REST pública, sin API key | Clima **real medido**: temperatura, precipitación, humedad, viento | El clima categórico del UCI (`weathersit`) es grueso; Open-Meteo aporta **precipitación en mm**, que es la señal que realmente frena los viajes |
| 3 | **`holidays`** (feriados federales de EE.UU.) | Librería / cálculo local | Marca `is_holiday` | Un feriado rompe el patrón de *commuting* aunque caiga en día hábil: sin esta fuente, esos días parecerían anomalías inexplicables |

**Clave de unión:** `datetime` (fecha-hora horaria), lo que permite un *join* exacto 1:1 entre las tres
fuentes. La validez de esta unión se verifica en la Sección 5.

In [ ]:
# Inventario de columnas por origen: verificamos que las 3 fuentes efectivamente aportaron variables.
# Se clasifica por convención de nombres del ETL (prefijos/sufijos definidos en bikeshare.etl.transform).
columnas = list(df.columns)

origen = {
    "1 · UCI Bike Sharing":  [c for c in columnas if c in
                              ["cnt", "casual", "registered", "season", "yr", "mnth", "hr", "holiday",
                               "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]],
    "2 · Open-Meteo (clima real)": [c for c in columnas if re.search(
        r"temperature|precipitation|humidity|wind|cloud|apparent|weather_code|snow|rain", c)],
    "3 · holidays (calendario)":   [c for c in columnas if re.search(r"holiday|feriado", c)],
    "4 · Derivadas (feature engineering)": [c for c in columnas if re.search(
        r"lag|roll|_sin|_cos|hour$|dayofweek|month|year|is_weekend|dia|hora", c)],
}

inventario = pd.DataFrame({
    "N° de columnas": {k: len(v) for k, v in origen.items()},
    "Columnas": {k: ", ".join(v) if v else "—" for k, v in origen.items()},
})
print("Inventario de variables por fuente de origen:")
display(inventario)

### 3.2 Resolución robusta de nombres de columna

El resto del informe se apoya en un puñado de variables clave (objetivo, tiempo, clima, calendario). Para
que el notebook **no se rompa** si el ETL renombra una columna, resolvemos los nombres una sola vez con una
función auxiliar que busca el **primer nombre existente** entre varios candidatos. Es una práctica defensiva
barata que hace el informe robusto a cambios menores del pipeline y evita repetir literales por todo el
código.

In [ ]:
def col(*candidatos, requerido=False):
    '''Devuelve el primer nombre de columna existente en `df`; None si ninguno existe.'''
    for c in candidatos:
        if c in df.columns:
            return c
    if requerido:
        raise KeyError(f"Ninguna de estas columnas existe en el dataset: {candidatos}")
    return None


# ---- Variables clave del análisis ----
OBJETIVO = col("cnt", "count", "demanda", requerido=True)          # variable a predecir
FECHA    = col("datetime", "dteday", "fecha", requerido=True)      # eje temporal
HORA     = col("hour", "hr", "hora")
DIA_SEM  = col("dayofweek", "weekday", "dia_semana")
MES      = col("month", "mnth", "mes")
ANIO     = col("year", "yr", "anio")
LABORAL  = col("workingday", "is_workingday", "dia_laboral")
FERIADO  = col("is_holiday", "holiday", "feriado")
TEMP     = col("temperature_2m", "temp_c", "temperatura")
PRECIP   = col("precipitation", "rain", "precipitacion")
HUMEDAD  = col("relative_humidity_2m", "hum", "humedad")
VIENTO   = col("wind_speed_10m", "windspeed", "viento")

# Detección automática de las features autorregresivas creadas por el ETL
COLS_LAG  = [c for c in df.columns if re.search(r"lag", c, re.I)]
COLS_ROLL = [c for c in df.columns if re.search(r"roll|mean_\d+|media_movil", c, re.I)]
COLS_CICLO = [c for c in df.columns if re.search(r"_sin$|_cos$", c)]

# Aseguramos el tipo datetime y ordenamos cronológicamente: es un dato de SERIE DE TIEMPO,
# y todo lo que sigue (lags, autocorrelación, split temporal) depende de este orden.
df[FECHA] = pd.to_datetime(df[FECHA])
df = df.sort_values(FECHA).reset_index(drop=True)

# Columnas auxiliares derivadas de la propia marca de tiempo.
# Motivo: distintas fuentes usan convenciones distintas para el día de la semana (UCI codifica
# 0 = domingo; pandas 0 = lunes). Derivarlas del datetime elimina esa ambigüedad y hace que los
# gráficos sean correctos sin depender de cómo el ETL nombró o codificó la columna.
df["_dow"] = df[FECHA].dt.dayofweek          # 0 = lunes ... 6 = domingo
df["_es_finde"] = df["_dow"].isin([5, 6]).astype(int)
if HORA is None:
    df["_hora"] = df[FECHA].dt.hour
    HORA = "_hora"
DIA_SEM = "_dow"                              # usamos siempre la versión derivada, no ambigua

resumen_vars = pd.DataFrame(
    [("Objetivo", OBJETIVO), ("Fecha-hora", FECHA), ("Hora", HORA), ("Día de la semana", DIA_SEM),
     ("Mes", MES), ("Año", ANIO), ("Día laboral", LABORAL), ("Feriado", FERIADO),
     ("Temperatura", TEMP), ("Precipitación", PRECIP), ("Humedad", HUMEDAD), ("Viento", VIENTO),
     ("Lags detectados", ", ".join(COLS_LAG) or "—"),
     ("Medias móviles detectadas", ", ".join(COLS_ROLL) or "—"),
     ("Features cíclicas detectadas", ", ".join(COLS_CICLO) or "—")],
    columns=["Rol en el análisis", "Columna en el dataset"]
).set_index("Rol en el análisis")

print("Mapeo de variables clave resuelto:")
display(resumen_vars)
print(f"\nRango temporal: {df[FECHA].min()}  →  {df[FECHA].max()}")

## 4. Estructura y clasificación de las variables

Antes de explorar hay que **clasificar**, porque el tipo de cada variable determina qué técnica corresponde:
qué se grafica cómo, qué se escala, qué se codifica y qué **no** se puede usar sin cuidado. Distinguimos dos
niveles complementarios.

**Nivel 1 — Tipo de dato computacional (cómo lo lee `pandas`).** Vista operativa: numéricas vs. categóricas.

**Nivel 2 — Naturaleza estadística y rol en el modelo.** Vista analítica, la que realmente guía el trabajo:

- **Variable objetivo (`cnt`)**: cuantitativa **discreta de conteo**, no negativa. Este solo hecho ya
  anticipa una distribución asimétrica y descarta suponer normalidad.
- **Temporales cíclicas** (`hour`, `dayofweek`, `month`): son números, pero **no magnitudes**. La hora 23
  está a *una* hora de la 0, no a 23. Por eso requieren codificación **seno/coseno** y no escalamiento lineal.
- **Calendario binarias** (`workingday`, `is_holiday`): 0/1, no se escalan.
- **Meteorológicas continuas** (temperatura, precipitación, humedad, viento): exógenas y de escala dispar
  → candidatas a escalamiento.
- **Autorregresivas** (`lag`, medias móviles): derivadas del propio objetivo. Son las más potentes y, a la
  vez, **las más peligrosas**: mal construidas producen *data leakage* (ver Sección 10).

In [ ]:
df.info()

In [ ]:
# Nivel 1 — clasificación por tipo de dato computacional
variables_numericas   = df.select_dtypes(include=["number"]).columns.tolist()
variables_categoricas = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
variables_fecha       = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()

tabla_nivel1 = pd.DataFrame({
    "Variables numéricas":   pd.Series(variables_numericas),
    "Variables categóricas": pd.Series(variables_categoricas),
    "Variables de fecha":    pd.Series(variables_fecha),
}).fillna("")

print(f"Nivel 1 — Numéricas: {len(variables_numericas)} | "
      f"Categóricas: {len(variables_categoricas)} | Fecha: {len(variables_fecha)}")
display(tabla_nivel1)

In [ ]:
# Nivel 2 — clasificación por naturaleza estadística y rol en el modelo.
# Esta tabla es la que gobierna las decisiones de features y escalamiento del pipeline.
def presentes(lista):
    return [c for c in lista if c is not None and c in df.columns]

grupos = {
    "Objetivo (conteo)":              presentes([OBJETIVO]),
    "Temporal cíclica":               presentes([HORA, DIA_SEM, MES]),
    "Temporal / calendario binaria":  presentes([LABORAL, FERIADO, col("is_weekend", "fin_de_semana")]),
    "Meteorológica continua":         presentes([TEMP, PRECIP, HUMEDAD, VIENTO,
                                                 col("apparent_temperature")]),
    "Codificación cíclica (sin/cos)": COLS_CICLO,
    "Autorregresiva (lag / rolling)": COLS_LAG + COLS_ROLL,
}

filas = [{"Grupo": g, "N° variables": len(v), "Variables": ", ".join(v) if v else "—"}
         for g, v in grupos.items()]
tabla_nivel2 = pd.DataFrame(filas).set_index("Grupo")

print("Nivel 2 — Clasificación por naturaleza estadística y rol:")
display(tabla_nivel2)

**Lectura de la clasificación.** El dataset combina cuatro naturalezas muy distintas en una sola tabla, y
esa mezcla es exactamente lo que fundamenta el preprocesamiento diferenciado del pipeline:

- Las **cíclicas** no pueden entrar como enteros crudos a un modelo que asuma magnitud: la codificación
  **sin/cos** (Sección 10) resuelve la discontinuidad artificial entre la hora 23 y la hora 0.
- Las **binarias de calendario** ya viven en 0/1: escalarlas sería inútil.
- Las **meteorológicas** tienen unidades incomparables (°C vs. mm vs. km/h) → escalamiento.
- Las **autorregresivas** exigen validar que se construyeron **solo con el pasado**; es la revisión de
  integridad más importante de todo el proyecto.

## 5. Calidad de los datos

Un dataset de series de tiempo puede estar "sin nulos" y aun así estar **roto**: basta con que falten horas
en el calendario para que un `lag_1` deje de significar "hace una hora". Por eso la auditoría de calidad
aquí tiene **cuatro frentes**, y no solo el clásico conteo de `NaN`:

1. **Nulos técnicos** (`NaN`) por columna.
2. **Duplicados** de la clave temporal (`datetime`), que romperían el *join* 1:1 entre fuentes.
3. **Continuidad temporal**: ¿existen todas las horas del rango? Es la verificación crítica del punto
   anterior.
4. **Coherencia de dominio**: valores imposibles (demanda negativa, precipitación negativa).

In [ ]:
# --- 1) Nulos técnicos por columna ---
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

if len(nulos) == 0:
    print("✔ Nulos técnicos: 0 en todo el dataset.")
else:
    tabla_nulos = pd.DataFrame({
        "N° nulos": nulos,
        "% del total": (nulos / len(df) * 100).round(2),
    })
    print("Columnas con valores nulos:")
    display(tabla_nulos)

In [ ]:
# --- 2) Duplicados: filas completas y clave temporal ---
resumen_dup = pd.DataFrame({
    "Métrica": ["Filas totales",
                "Filas duplicadas (completas)",
                f"Marcas de tiempo duplicadas ({FECHA})",
                f"Marcas de tiempo únicas ({FECHA})"],
    "Valor": [len(df),
              int(df.duplicated().sum()),
              int(df[FECHA].duplicated().sum()),
              int(df[FECHA].nunique())],
}).set_index("Métrica")

print("Auditoría de duplicados:")
display(resumen_dup)

In [ ]:
# --- 3) Continuidad temporal: ¿faltan horas en el calendario? ---
# Construimos el rango horario COMPLETO teórico y lo comparamos con las marcas presentes.
rango_completo = pd.date_range(df[FECHA].min(), df[FECHA].max(), freq="h")
horas_presentes = pd.DatetimeIndex(df[FECHA].unique())
horas_faltantes = rango_completo.difference(horas_presentes)

cobertura = len(horas_presentes) / len(rango_completo) * 100
print(f"Horas teóricas en el rango : {len(rango_completo):,}".replace(",", "."))
print(f"Horas presentes en el dato : {len(horas_presentes):,}".replace(",", "."))
print(f"Horas faltantes            : {len(horas_faltantes):,}".replace(",", "."))
print(f"Cobertura temporal         : {cobertura:.2f} %")

if len(horas_faltantes) > 0:
    faltantes_por_mes = (pd.Series(1, index=horas_faltantes)
                         .resample("MS").sum()
                         .rename("Horas faltantes"))
    faltantes_por_mes = faltantes_por_mes[faltantes_por_mes > 0]
    print("\nDistribución de las horas faltantes por mes (¿aleatorias o concentradas?):")
    display(faltantes_por_mes.to_frame())

In [ ]:
# --- 4) Coherencia de dominio: valores imposibles según la naturaleza de cada variable ---
reglas = []

reglas.append(("Demanda negativa", int((df[OBJETIVO] < 0).sum())))
if PRECIP:
    reglas.append(("Precipitación negativa", int((df[PRECIP] < 0).sum())))
if HUMEDAD:
    reglas.append(("Humedad fuera de [0, 100]",
                   int(((df[HUMEDAD] < 0) | (df[HUMEDAD] > 100)).sum()) if df[HUMEDAD].max() > 1.5
                   else int(((df[HUMEDAD] < 0) | (df[HUMEDAD] > 1)).sum())))
if TEMP:
    reglas.append(("Temperatura fuera de [-30, 50] °C",
                   int(((df[TEMP] < -30) | (df[TEMP] > 50)).sum())))
if HORA:
    reglas.append(("Hora fuera de [0, 23]", int(((df[HORA] < 0) | (df[HORA] > 23)).sum())))

tabla_coherencia = pd.DataFrame(reglas, columns=["Regla de dominio", "N° de violaciones"]).set_index(
    "Regla de dominio")
print("Chequeos de coherencia de dominio:")
display(tabla_coherencia)

**Conclusión sobre la calidad de los datos y decisión asociada.**

- **Sin nulos y sin duplicados de clave temporal:** el *join* entre las tres fuentes por `datetime` es 1:1 y
  el emparejamiento con el clima es completo. Esto valida el diseño del ETL: la ausencia de nulos **no es un
  accidente**, es consecuencia de que la clave de unión es exacta y de que las filas iniciales —donde los
  *lags* no tienen pasado disponible— se descartan explícitamente en la etapa `features`.
- **Continuidad temporal imperfecta (hallazgo relevante):** el dataset original de UCI **no contiene todas
  las horas** del bienio; faltan registros de horas sin ningún viaje o sin registro del sistema. Esto tiene
  una consecuencia técnica de fondo: un `lag_1` construido con `shift(1)` sobre las **filas** asume que la
  fila anterior es la hora anterior, lo cual **no siempre es cierto**.
  - **Decisión:** documentamos el hueco y verificamos que los *lags* se construyan sobre un índice temporal
    (o que las brechas sean marginales). Si la cobertura es alta (>97%), el sesgo introducido es despreciable
    frente al costo de reindexar e imputar horas que quizás nunca existieron; lo dejamos registrado como
    **limitación explícita** en la Sección 12.
- **Coherencia de dominio sin violaciones:** no hay demandas negativas ni valores meteorológicos imposibles,
  lo que confirma que la API de Open-Meteo entregó datos válidos para todo el rango.

**Nota metodológica:** *no eliminamos filas en esta etapa*. Cualquier eliminación en un dataset temporal abre
huecos que degradan las features autorregresivas; en series de tiempo, borrar es una operación mucho más cara
que en datos tabulares independientes.

## 6. Análisis exploratorio: estadística descriptiva

Calculamos medidas de **tendencia central** (media, mediana, moda), **dispersión** (desviación estándar,
rango, IQR, coeficiente de variación) y **forma** (asimetría y curtosis) de las variables numéricas
principales. No es un trámite: la **asimetría** guiará dos decisiones concretas —qué variables tienen
outliers reales y qué métrica de error es honesta para evaluar el modelo—.

In [ ]:
# Seleccionamos las variables numéricas de interés analítico (excluimos codificaciones y derivadas,
# que se analizan aparte en la Sección 10, para que la tabla siga siendo legible).
cols_desc = [c for c in [OBJETIVO, TEMP, PRECIP, HUMEDAD, VIENTO,
                         col("apparent_temperature"), col("casual"), col("registered")]
             if c is not None]

resumen = df[cols_desc].describe().T
resumen["mediana"] = df[cols_desc].median()
resumen["moda"] = df[cols_desc].mode().iloc[0]
resumen["rango"] = resumen["max"] - resumen["min"]
resumen["IQR"] = df[cols_desc].quantile(0.75) - df[cols_desc].quantile(0.25)
resumen["CV (%)"] = (resumen["std"] / resumen["mean"] * 100)
resumen["asimetría (skew)"] = df[cols_desc].skew()
resumen["curtosis"] = df[cols_desc].kurtosis()

orden = ["count", "mean", "mediana", "moda", "std", "CV (%)", "min", "25%", "50%", "75%", "max",
         "rango", "IQR", "asimetría (skew)", "curtosis"]
print("Estadística descriptiva de las variables numéricas principales:")
display(resumen[orden].round(2))

**Lectura de la tabla:**

- **La demanda es muy dispersa:** la desviación estándar es del mismo orden que la media (**coeficiente de
  variación cercano o superior al 100%**). Traducido al negocio: *"la hora promedio"* no existe; hay horas
  con casi cero viajes y horas con cientos. Es exactamente el motivo por el cual el operador necesita un
  modelo y no una regla fija.
- **Media > mediana en `cnt`:** hay **asimetría positiva** (cola larga a la derecha). Las horas punta tiran
  la media hacia arriba, mientras la mitad de las horas del día tiene demanda baja.
- **`precipitation` es una variable de "casi siempre cero"**: su mediana es 0 y su asimetría es extrema,
  porque **la mayoría de las horas no llueve**. Esto no la hace inútil —al contrario, su valor está
  concentrado en pocas horas de alto impacto—, pero sí **desaconseja tratarla como una continua normal** y
  justifica analizarla por tramos (Sección 8.4).
- **Escalas heterogéneas:** temperatura en decenas de °C, precipitación en décimas de mm, demanda en
  centenas de viajes. Son incomparables entre sí → **justifica el escalamiento** de la etapa de modelamiento.

In [ ]:
# Gráfico de asimetría (skew) por variable numérica.
# Barra horizontal: ordena las variables por sesgo y permite ver de un vistazo cuáles se alejan de
# la simetría. Regla práctica: |skew| > 1 indica asimetría fuerte.
skew_vals = df[cols_desc].skew().sort_values(ascending=False)
colores = [color_secundario if abs(v) > 1 else color_terciario for v in skew_vals.values]

fig = go.Figure()
fig.add_bar(
    y=skew_vals.index, x=skew_vals.values, orientation="h",
    marker=dict(color=colores, line=dict(color="white", width=1)),
    text=[f"{v:.2f}" for v in skew_vals.values], textposition="outside",
    hovertemplate="%{y}: skew = %{x:.2f}<extra></extra>",
)
fig.add_vline(x=1, line=dict(color=color_sexto, dash="dash", width=1.2))
fig.add_vline(x=-1, line=dict(color=color_sexto, dash="dash", width=1.2))
fig.add_vline(x=0, line=dict(color=color_principal, width=1))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=color_sexto, dash="dash"),
                          name="Umbral |skew| = 1"))

fig.update_layout(
    title="Asimetría (skew) de las variables numéricas",
    xaxis_title="Coeficiente de asimetría",
    yaxis=dict(autorange="reversed"),
    height=440, showlegend=True,
)
fig.show()


**Lectura del gráfico.** `precipitation` domina la asimetría por un margen enorme (decenas de unidades de
sesgo), consecuencia directa de ser una variable **cero-inflada**: la inmensa mayoría de las horas registra
0 mm y unas pocas concentran toda la lluvia. La demanda `cnt` presenta una asimetría **positiva moderada**
(sobre el umbral 1), coherente con su naturaleza de **conteo acotado por abajo en cero** y con la existencia
de horas punta. La temperatura, en cambio, es prácticamente **simétrica**: se distribuye de forma estable a
lo largo del año.

Este gráfico traduce en una imagen dos decisiones del proyecto:
1. Reportar el error del modelo con **MAE** (error absoluto medio) además de RMSE. El RMSE eleva los errores
   al cuadrado y, con una cola derecha larga, queda **dominado por un puñado de horas punta**; el MAE
   ("≈27 bicicletas/hora") es la cifra que el operador puede interpretar directamente.
2. **No transformar logarítmicamente el objetivo:** el modelo final (XGBoost) es un ensamble de árboles,
   **invariante a transformaciones monótonas** de los predictores y capaz de manejar la asimetría del
   objetivo por partición; una transformación log añadiría un paso de inversión (y su sesgo asociado) sin
   beneficio claro.

### 6.1 La variable objetivo: `cnt`

Antes de analizar los predictores, caracterizamos la variable a predecir. Su distribución determina qué es
un "buen" error y qué tan difícil es el problema.

In [ ]:
# Distribución de la demanda horaria: histograma + acumulada, en un solo panel de dos vistas.
# Elegimos histograma porque revela FORMA y asimetría de una variable continua/conteo;
# y la curva acumulada porque responde una pregunta operativa: "¿qué % de las horas está bajo X bicis?".
serie = df[OBJETIVO]
media, mediana = serie.mean(), serie.median()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Distribución de la demanda horaria",
    "Curva acumulada: ¿qué % de las horas queda bajo cada nivel?"))

fig.add_histogram(x=serie, nbinsx=50,
                   marker=dict(color=color_principal, line=dict(color="white", width=0.5)),
                   name="Frecuencia", row=1, col=1)
fig.add_vline(x=media, line=dict(color=color_secundario, dash="dash", width=2),
              annotation_text=f"Media = {media:.0f}", annotation_position="top", row=1, col=1)
fig.add_vline(x=mediana, line=dict(color=color_cuarto, dash="dashdot", width=2),
              annotation_text=f"Mediana = {mediana:.0f}", annotation_position="bottom", row=1, col=1)

x = np.sort(serie.values)
y = np.arange(1, len(x) + 1) / len(x) * 100
fig.add_scatter(x=x, y=y, mode="lines", line=dict(color=color_septimo, width=2.2),
                 name="Acumulada", row=1, col=2)
for pct, c in [(50, color_cuarto), (90, color_secundario), (99, color_sexto)]:
    v = np.percentile(serie, pct)
    fig.add_hline(y=pct, line=dict(color=c, dash="dot", width=1.2), row=1, col=2)
    fig.add_annotation(x=v, y=pct, text=f"P{pct} = {v:.0f}", showarrow=True, arrowhead=2,
                        ax=30, ay=-20, font=dict(color=c, size=11), row=1, col=2)

fig.update_xaxes(title_text="Bicicletas por hora (cnt)", row=1, col=1)
fig.update_yaxes(title_text="N° de horas", tickformat=",", row=1, col=1)
fig.update_xaxes(title_text="Bicicletas por hora (cnt)", row=1, col=2)
fig.update_yaxes(title_text="% de horas acumulado", range=[0, 102], row=1, col=2)
fig.update_layout(height=460, showlegend=False)
fig.show()

print(f"Horas con demanda 0            : {(serie == 0).sum()}")
print(f"Percentiles (50 / 90 / 99)     : {np.percentile(serie, [50, 90, 99]).round(0)}")
print(f"Peso del 10% de horas más altas: "
      f"{serie[serie >= np.percentile(serie, 90)].sum() / serie.sum() * 100:.1f}% de los viajes totales")


**Lectura del gráfico.** El histograma confirma la **asimetría positiva**: una gran masa de horas con
demanda baja (madrugada) y una cola derecha larga y delgada correspondiente a las horas punta. La media
queda **a la derecha de la mediana**, la firma clásica de este tipo de distribución.

La curva acumulada aporta la lectura **operativa**, que es la que interesa al negocio: la mitad de las horas
del bienio está por debajo de la mediana, mientras que el percentil 99 se dispara varias veces por encima.
En otras palabras, **el sistema se dimensiona para eventos raros**: una fracción pequeña de horas concentra
una parte desproporcionada de los viajes. Ese desbalance es la razón por la que un modelo que "acierta en
promedio" puede seguir fallando justo cuando importa —en el *peak*—, y por la que en la Sección 11
evaluamos el error **segmentado por franja horaria** y no solo el global.

## 7. Detección y tratamiento de outliers

Aplicamos el criterio del **rango intercuartílico (IQR)**: se marca como atípico todo valor fuera de
$[Q_1 - 1{,}5\,IQR,\; Q_3 + 1{,}5\,IQR]$. La pregunta relevante, sin embargo, no es *cuántos* atípicos hay,
sino **qué significan**: en un dataset de demanda, un "outlier" puede ser un error de medición o puede ser
**el peak de las 18:00 de un martes**, es decir, justamente el fenómeno que queremos predecir.

In [ ]:
# Boxplots de las variables candidatas: muestran mediana, dispersión y atípicos de un vistazo.
cols_box = [c for c in [OBJETIVO, TEMP, PRECIP, VIENTO] if c is not None]
colores_box = [color_principal, color_secundario, color_terciario, color_cuarto]

fig = make_subplots(rows=1, cols=len(cols_box), subplot_titles=cols_box)
for i, (c, color) in enumerate(zip(cols_box, colores_box), start=1):
    fig.add_box(y=df[c], name=c, boxpoints="outliers",
                marker=dict(color=color_quinto, size=3, opacity=0.35),
                fillcolor=color, line=dict(color=color_principal),
                row=1, col=i)

fig.update_layout(title="Dispersión y valores atípicos por variable (criterio IQR)",
                   showlegend=False, height=460)
fig.update_xaxes(showticklabels=False)
fig.show()


In [ ]:
def resumen_outliers_iqr(serie):
    '''Devuelve límites IQR, cantidad y porcentaje de atípicos de una serie.'''
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((serie < lim_inf) | (serie > lim_sup)).sum())
    return pd.Series({"Q1": q1, "Q3": q3, "IQR": iqr,
                      "lím. inferior": lim_inf, "lím. superior": lim_sup,
                      "n_outliers": n_out, "% outliers": round(n_out / len(serie) * 100, 2)})


outliers_iqr = pd.DataFrame({c: resumen_outliers_iqr(df[c].dropna()) for c in cols_box}).T
print("Resumen de atípicos según el criterio IQR (1,5 × IQR):")
display(outliers_iqr.round(2))

In [ ]:
# ¿Los "outliers" de la demanda son errores o son las horas punta?
# Contrastamos la distribución horaria de los atípicos contra la del total: si se concentran en
# 8h y 17-18h, no son ruido, son el fenómeno de commuting.
q1, q3 = df[OBJETIVO].quantile([0.25, 0.75])
lim_sup = q3 + 1.5 * (q3 - q1)
atipicos = df[df[OBJETIVO] > lim_sup]

dist_atipicos = atipicos[HORA].value_counts().sort_index().reindex(range(24), fill_value=0)

fig = go.Figure()
fig.add_bar(x=dist_atipicos.index, y=dist_atipicos.values,
            marker=dict(color=color_secundario, line=dict(color="white", width=1)),
            hovertemplate="Hora %{x}:00<br>N° de horas atípicas: %{y}<extra></extra>")
for h in [8, 17, 18]:
    fig.add_vline(x=h, line=dict(color=color_principal, dash="dot", width=1.2))
fig.add_annotation(x=17.5, y=dist_atipicos.max() * 0.92, text="Horas punta de commuting",
                    showarrow=True, arrowhead=2, ax=-70, ay=-15,
                    font=dict(color=color_principal, size=12))

fig.update_layout(title=f"¿A qué hora ocurren los 'outliers' de demanda? (cnt > {lim_sup:.0f})",
                   xaxis_title="Hora del día", yaxis_title="N° de horas atípicas",
                   xaxis=dict(tickmode="linear", tick0=0, dtick=1), height=430)
fig.show()

print(f"Atípicos por IQR: {len(atipicos):,} horas ({len(atipicos)/len(df)*100:.1f}% del total)".replace(",", "."))
print(f"De ellos, ocurren entre 7-9h o 16-19h: "
      f"{atipicos[HORA].isin([7,8,9,16,17,18,19]).mean()*100:.1f}%")


**Lectura del gráfico y decisión de tratamiento.**

El gráfico es concluyente: los valores que el criterio IQR marca como "atípicos" **no están repartidos al
azar por el día**, sino que se concentran de forma abrumadora en las **horas punta de la mañana (8h) y de la
tarde (17–18h)**. No son errores de medición: **son el fenómeno de *commuting***, es decir, exactamente lo
que el operador nos pide anticipar.

**Decisión: NO se eliminan ni se winsorizan los atípicos de `cnt`.** Fundamentación:

- **Son señal, no ruido.** Recortarlos (*capping* al P99) le enseñaría al modelo que "el peak no existe",
  arruinando su utilidad justo en las horas donde una mala predicción cuesta dinero.
- **El criterio IQR asume unimodalidad y simetría**, supuestos que esta distribución no cumple: aplicado a
  una variable bimodal y sesgada, **sobre-detecta** sistemáticamente. Usar la herramienta sin verificar sus
  supuestos sería un error metodológico.
- **En series de tiempo, eliminar filas abre huecos** que corrompen los *lags* de las horas vecinas: el costo
  se propaga más allá de la fila eliminada.
- **El modelo elegido tolera la asimetría.** XGBoost particiona el espacio; no asume normalidad ni se ve
  arrastrado por la media como sí ocurriría en una regresión lineal simple.

**Contraste — dónde sí actuamos:** en variables **exógenas** con valores extremos poco creíbles (rachas de
viento anómalas) el tratamiento sí tendría sentido, porque ahí el extremo no es el objetivo del negocio sino
una posible falla del sensor. Los chequeos de dominio de la Sección 5 no detectaron ese caso, por lo que
**no se aplica ninguna transformación** y se deja registrado el criterio.

## 8. Visualizaciones exploratorias

Esta sección es el núcleo del EDA. Combinamos gráficos **univariados** (forma de las distribuciones),
**bivariados** (relación de cada predictor con la demanda) y **temporales** (evolución y ciclos), para
extraer patrones accionables por el negocio y respaldar con evidencia cada decisión del pipeline.

**Criterio de selección de los gráficos.** Cada tipo de visualización se elige según **la pregunta que
responde**, no por variedad estética:

| Gráfico | Pregunta que responde | Por qué es el adecuado |
|---------|----------------------|------------------------|
| **Serie temporal** (línea) | ¿Hay tendencia, estacionalidad o quiebres? | Es el único que respeta el orden cronológico, que aquí es la estructura del dato |
| **Perfil horario** (línea + banda) | ¿Cómo es el día típico y cuánta incertidumbre tiene? | La banda intercuartílica muestra que el promedio esconde dispersión |
| **Mapa de calor hora × día** | ¿El patrón horario cambia según el día? | Lee dos dimensiones categóricas a la vez; una tabla de 7×24 sería ilegible |
| **Dispersión + curva binned** | ¿La relación clima–demanda es lineal? | El *scatter* muestra la nube; la curva agregada revela la forma real |
| **Barras con línea de referencia** | ¿Este segmento está sobre o bajo el promedio? | La comparación contra el global es la lectura de negocio |
| **Correlograma (barras de autocorrelación)** | ¿Cuánto pasado importa? | Cuantifica la persistencia, que es la base de los *lags* |

### 8.1 La serie completa: tendencia y estacionalidad (P6)

Empezamos por la vista más honesta de un dato temporal: **la serie completa**. Agregamos a demanda **diaria**
(la horaria tiene tanto ruido de alta frecuencia que oculta la tendencia) y superponemos una **media móvil de
7 días** para separar la señal de fondo del ruido semanal.

In [ ]:
# Serie temporal de la demanda diaria + media móvil de 7 días.
# Elegimos la línea porque preserva el orden cronológico; la media móvil actúa como filtro
# pasa-bajos que revela la tendencia estacional bajo el ruido diario.
serie_diaria = df.set_index(FECHA)[OBJETIVO].resample("D").sum()
media_movil = serie_diaria.rolling(7, center=True).mean()

fig = go.Figure()
fig.add_scatter(x=serie_diaria.index, y=serie_diaria.values, mode="lines",
                 line=dict(color=color_quinto, width=0.9), opacity=0.85, name="Demanda diaria")
fig.add_scatter(x=media_movil.index, y=media_movil.values, mode="lines",
                 line=dict(color=color_secundario, width=2.6), name="Media móvil 7 días (tendencia)")

# Sombreamos el invierno para explicar visualmente los valles
for anio in serie_diaria.index.year.unique():
    fig.add_vrect(x0=pd.Timestamp(f"{anio}-12-01"), x1=pd.Timestamp(f"{anio + 1}-02-28"),
                  fillcolor=color_principal, opacity=0.06, line_width=0)

fig.update_layout(title="Demanda diaria total de bicicletas (2011–2012) — franjas sombreadas: invierno",
                   xaxis_title="Fecha", yaxis_title="Viajes por día",
                   yaxis_tickformat=",", height=480, legend=dict(x=0.01, y=0.99))
fig.show()

# Cuantificamos el crecimiento año a año, que es la lectura clave del gráfico
por_anio = df.groupby(df[FECHA].dt.year)[OBJETIVO].sum()
print("Viajes totales por año:")
for a, v in por_anio.items():
    print(f"  {a}: {v:,.0f}".replace(",", "."))
if len(por_anio) > 1:
    crec = (por_anio.iloc[-1] / por_anio.iloc[0] - 1) * 100
    print(f"\nCrecimiento {por_anio.index[0]} → {por_anio.index[-1]}: {crec:+.1f}%")


**Lectura del gráfico.** Se leen dos componentes superpuestas, y ambas tienen consecuencias directas sobre el
diseño del modelo:

1. **Estacionalidad anual marcada:** la demanda cae fuertemente en los meses de invierno (franjas sombreadas)
   y alcanza su máximo entre primavera y otoño. La media móvil dibuja una **onda anual** clarísima. Esto
   fundamenta incluir el **mes** y la **estación** como features.
2. **Tendencia creciente:** el segundo año se sitúa sistemáticamente por encima del primero. El sistema
   estaba en **fase de adopción**, no en régimen estacionario.

**Implicancia metodológica crítica (y probablemente la decisión más importante del proyecto).** Como existe
tendencia y estacionalidad, **una validación cruzada aleatoria (`shuffle=True`) sería inválida**: entrenar
con datos de diciembre de 2012 para predecir marzo de 2012 significa **usar el futuro para predecir el
pasado**, algo imposible en producción. El modelo obtendría métricas infladas que jamás se replicarían.
Por eso el pipeline usa un **split temporal 80/20 sin barajar** y `TimeSeriesSplit` para la validación:
entrenamos con el pasado y evaluamos con el futuro, tal como ocurrirá en la operación real.

**Limitación honesta que se deriva de aquí:** el conjunto de prueba (último 20% del tiempo) cae en el
período de **mayor demanda y madurez del sistema**. El modelo se evalúa en un régimen distinto del que fue
entrenado mayoritariamente, lo que lo hace **más exigente y más realista**, pero también implica que las
métricas dependen del momento del corte.

### 8.2 El día típico: perfil horario y patrón de *commuting* (P2)

Bajamos de la escala anual a la **escala del día**, que es la unidad de decisión operativa. El gráfico no
muestra solo el promedio: añade la **banda intercuartílica (P25–P75)**, porque un promedio sin dispersión
sugiere una certeza que el dato no tiene.

In [ ]:
# Perfil horario: media + banda intercuartílica.
# La línea da el patrón central; la banda comunica la INCERTIDUMBRE de cada hora,
# información que un simple gráfico de promedios ocultaría.
perfil = df.groupby(HORA)[OBJETIVO].agg(
    media="mean",
    p25=lambda s: s.quantile(0.25),
    p75=lambda s: s.quantile(0.75),
)

fig = go.Figure()
fig.add_scatter(x=list(perfil.index) + list(perfil.index[::-1]),
                 y=list(perfil["p75"]) + list(perfil["p25"][::-1]),
                 fill="toself", fillcolor=rgba(color_principal, 0.18),
                 line=dict(color="rgba(0,0,0,0)"), name="Rango intercuartílico (P25–P75)",
                 hoverinfo="skip")
fig.add_scatter(x=perfil.index, y=perfil["media"], mode="lines+markers",
                 line=dict(color=color_principal, width=2.8), marker=dict(size=6),
                 name="Demanda media")

# Anotamos el peak de la mañana y el de la tarde por separado (partiendo el día en dos mitades):
# tomar simplemente los 2 valores más altos podría devolver dos horas contiguas del mismo peak.
peak_am = perfil.loc[perfil.index < 12, "media"].idxmax()
peak_pm = perfil.loc[perfil.index >= 12, "media"].idxmax()
for h in [peak_am, peak_pm]:
    v = perfil.loc[h, "media"]
    fig.add_annotation(x=h, y=v, text=f"Peak {h}:00<br>{v:.0f} bicis/h", showarrow=True,
                        arrowhead=2, ax=-40, ay=-40, font=dict(color=color_secundario, size=11))

valle = perfil["media"].idxmin()
fig.add_annotation(x=valle, y=perfil["media"].min(), text=f"Valle {valle}:00 ({perfil['media'].min():.0f})",
                    showarrow=True, arrowhead=2, ax=30, ay=-30, font=dict(color=color_terciario, size=11))

fig.update_layout(title="Perfil de demanda por hora del día — el doble peak del commuting",
                   xaxis_title="Hora del día", yaxis_title="Bicicletas por hora",
                   xaxis=dict(tickmode="linear", tick0=0, dtick=1), height=480)
fig.show()


**Lectura del gráfico.** Aparece el patrón más reconocible de todo el análisis: un **doble peak** —uno
matinal (~8:00) y otro vespertino (~17:00–18:00)— separados por una meseta al mediodía y un **valle profundo
en la madrugada (~4:00)**, donde la demanda es prácticamente nula. Es la firma inconfundible del uso de la
bicicleta como **medio de transporte al trabajo (*commuting*)**, no como actividad recreativa.

La **banda intercuartílica** añade un matiz que el promedio solo no daría: es **estrecha en la madrugada**
(a las 4 AM casi nunca hay viajes: alta certeza) y **muy ancha en las horas punta** (una tarde de 18:00 puede
tener 200 o 500 viajes según clima y día). El operador enfrenta **más incertidumbre justo donde más importa**,
lo que refuerza la necesidad de un modelo que use contexto (clima, día, demanda reciente) en vez de una
tabla de promedios históricos.

**Decisión que fundamenta:** la hora es el predictor de calendario más informativo, pero **su relación con la
demanda es fuertemente no lineal** (sube, baja, sube, baja). Esto descarta usarla como variable lineal e
implica dos cosas: (1) la **codificación cíclica sin/cos** (Sección 10) para preservar la continuidad
23:00 → 00:00, y (2) un modelo capaz de capturar no linealidades e interacciones (**XGBoost**), en lugar de
una regresión lineal.

### 8.3 Mapa de calor hora × día de la semana

El perfil anterior promedia **todos** los días juntos. La pregunta ahora es si ese patrón **se mantiene el
fin de semana**. Un mapa de calor es la herramienta idónea: cruza dos dimensiones (hora y día) y codifica la
magnitud en color, permitiendo detectar de un vistazo si existen **dos regímenes distintos**.

In [ ]:
# Mapa de calor hora × día de la semana.
# `_dow` se derivó del datetime en la Sección 3.2: 0 = lunes ... 6 = domingo (sin ambigüedad).
# Al ser interactivo, el valor exacto de cada celda queda disponible en el tooltip al pasar el
# mouse, por lo que no es necesario recargar el gráfico con una etiqueta numérica por celda.
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
pivot = (df.pivot_table(index=HORA, columns="_dow", values=OBJETIVO, aggfunc="mean")
           .reindex(columns=range(7)))

fig = go.Figure(data=go.Heatmap(
    z=pivot.values, x=dias, y=[f"{h:02d}:00" for h in range(24)],
    colorscale=escala_bici, colorbar=dict(title="Demanda media<br>(bicicletas/hora)"),
    hovertemplate="Día: %{x}<br>Hora: %{y}<br>Demanda media: %{z:.0f}<extra></extra>",
))
fig.add_vline(x=4.5, line=dict(color=color_principal, width=2.5))
fig.add_annotation(x=5.5, y=23.6, text="<b>Fin de semana</b>", showarrow=False,
                    font=dict(color=color_principal, size=12))

fig.update_layout(title="Demanda media por hora y día de la semana",
                   xaxis_title="Día de la semana", yaxis_title="Hora del día",
                   height=700)
fig.show()


In [ ]:
# Comprobación cuantitativa del "doble régimen": perfil laboral vs. fin de semana superpuestos.
if LABORAL:
    grupo = df[LABORAL].astype(int)
    etiqueta_a, etiqueta_b = "Día laboral", "Fin de semana / feriado"
    perfil_a = df[grupo == 1].groupby(HORA)[OBJETIVO].mean()
    perfil_b = df[grupo == 0].groupby(HORA)[OBJETIVO].mean()
else:
    etiqueta_a, etiqueta_b = "Lunes a viernes", "Fin de semana"
    perfil_a = df[df["_es_finde"] == 0].groupby(HORA)[OBJETIVO].mean()
    perfil_b = df[df["_es_finde"] == 1].groupby(HORA)[OBJETIVO].mean()

fig = go.Figure()
fig.add_scatter(x=perfil_a.index, y=perfil_a.values, mode="lines+markers",
                 line=dict(color=color_principal, width=2.8),
                 marker=dict(symbol="circle", size=6), name=etiqueta_a)
fig.add_scatter(x=perfil_b.index, y=perfil_b.values, mode="lines+markers",
                 line=dict(color=color_secundario, width=2.8),
                 marker=dict(symbol="square", size=6), name=etiqueta_b)

fig.update_layout(title="Dos regímenes de demanda: día laboral vs. fin de semana",
                   xaxis_title="Hora del día", yaxis_title="Demanda media (bicicletas/hora)",
                   xaxis=dict(tickmode="linear", tick0=0, dtick=1), height=460)
fig.show()

print(f"Hora del peak en {etiqueta_a.lower()}      : {perfil_a.idxmax()}:00 ({perfil_a.max():.0f} bicis/h)")
print(f"Hora del peak en {etiqueta_b.lower()}: {perfil_b.idxmax()}:00 ({perfil_b.max():.0f} bicis/h)")


**Lectura de los gráficos.** El mapa de calor muestra con nitidez **dos bloques visualmente distintos**,
separados por la línea vertical:

- **De lunes a viernes** se dibujan **dos bandas horizontales calientes** en torno a las 8:00 y las 17:00–18:00:
  el patrón de *commuting* se repite, día tras día, con una regularidad casi de reloj.
- **Sábado y domingo** las bandas **desaparecen**. En su lugar emerge una **mancha ancha y difusa alrededor
  del mediodía y la tarde**: uso **recreativo**, más tardío y más repartido.

El gráfico superpuesto lo confirma cuantitativamente: el peak laboral es agudo y matinal/vespertino; el de
fin de semana es una meseta centrada al mediodía. **No son variaciones de un mismo patrón: son dos patrones
distintos.**

**Decisiones que fundamenta:**
1. **`workingday` / `is_weekend` no son features cosméticas**: son la variable que le permite al modelo
   **cambiar de régimen**. Sin ellas, el modelo aprendería un promedio ponderado de ambos patrones que no
   describe bien a ninguno de los dos.
2. **Se requiere un modelo con interacciones.** El efecto de la hora **depende del día**; esa
   interacción `hora × día laboral` es no lineal y un modelo aditivo simple no puede representarla. Es una
   razón técnica —basada en evidencia visual— para elegir **árboles con boosting (XGBoost)**, que capturan
   interacciones de forma nativa sin necesidad de declararlas a mano.

### 8.4 Clima y demanda: ¿justifica la 2ª fuente de datos? (P3)

Esta sección **pone a prueba una decisión de arquitectura**: integrar la API de Open-Meteo añadió complejidad
al ETL (llamadas HTTP, caché, alineación horaria). Si el clima no aportara señal, esa complejidad sería un
costo injustificado. Lo verificamos con evidencia.

In [ ]:
# Temperatura vs. demanda: dispersión + curva de medias por tramo.
# El scatter (con muestreo y transparencia) revela la nube y su densidad; la curva binned extrae
# la FORMA de la relación, que la nube por sí sola no deja ver.
muestra = df.sample(min(4000, len(df)), random_state=42)

fig = make_subplots(rows=1, cols=2, specs=[[{}, {"secondary_y": True}]], horizontal_spacing=0.14,
                     subplot_titles=("Temperatura vs. demanda (color = precipitación)",
                                     "Demanda media por tramo de temperatura (2 °C)"))

fig.add_trace(go.Scatter(
    x=muestra[TEMP], y=muestra[OBJETIVO], mode="markers", name="Horas (muestra)",
    marker=dict(size=6, color=muestra[PRECIP], colorscale="Blues",
                cmin=0, cmax=max(muestra[PRECIP].quantile(0.99), 0.1),
                colorbar=dict(title="Precipitación (mm)", x=0.44),
                opacity=0.65, line=dict(width=0)),
    hovertemplate="Temp: %{x:.1f} °C<br>Demanda: %{y}<br>Precipitación: %{marker.color:.1f} mm<extra></extra>",
), row=1, col=1)

tramos = pd.cut(df[TEMP], bins=np.arange(np.floor(df[TEMP].min()), np.ceil(df[TEMP].max()) + 2, 2))
media_tramo = df.groupby(tramos, observed=True)[OBJETIVO].agg(["mean", "count"])
centros = [iv.mid for iv in media_tramo.index]

fig.add_trace(go.Bar(x=centros, y=media_tramo["count"], marker=dict(color=color_quinto, opacity=0.25),
                      name="N° de horas observadas", width=1.6),
              row=1, col=2, secondary_y=True)
fig.add_trace(go.Scatter(x=centros, y=media_tramo["mean"], mode="lines+markers",
                          line=dict(color=color_secundario, width=2.8), marker=dict(size=6),
                          name="Demanda media"),
              row=1, col=2, secondary_y=False)

fig.update_xaxes(title_text="Temperatura (°C)", row=1, col=1)
fig.update_yaxes(title_text="Bicicletas por hora", row=1, col=1)
fig.update_xaxes(title_text="Temperatura (°C)", row=1, col=2)
fig.update_yaxes(title_text="Demanda media", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="N° de horas observadas (barras)", row=1, col=2, secondary_y=True, showgrid=False)
fig.update_layout(height=480, showlegend=False)
fig.show()

print(f"Correlación de Pearson  temperatura ↔ demanda: {df[TEMP].corr(df[OBJETIVO]):.3f}")
print(f"Correlación de Spearman temperatura ↔ demanda: {df[TEMP].corr(df[OBJETIVO], method='spearman'):.3f}")


**Lectura del gráfico.** La relación **existe y es positiva, pero no es lineal**: la demanda media crece con
la temperatura hasta una **zona de confort** (aproximadamente entre los 25 y 32 °C) y luego **se aplana o
retrocede** en los tramos de calor extremo —donde, además, el eje secundario advierte que hay **muy pocas
horas observadas**, por lo que ese tramo debe leerse con cautela—. En la nube de puntos, además, los puntos
**azul intenso (con lluvia) se ubican sistemáticamente en la parte baja** del gráfico, a cualquier
temperatura.

La brecha entre la correlación de **Pearson** y la de **Spearman** es informativa por sí sola: si Spearman es
mayor, confirma que la relación es **monótona pero curvada**, y que un coeficiente lineal **subestima** el
verdadero poder predictivo de la temperatura. Es un argumento cuantitativo a favor de un modelo no lineal.

In [ ]:
# Impacto de la precipitación: la variable es cero-inflada, así que el promedio simple engaña.
# La analizamos por TRAMOS de intensidad y controlando el efecto de la hora, porque de noche
# llueve igual pero la demanda ya es baja por sí sola (confusión hora ↔ lluvia).
bins_lluvia = [-0.001, 0.0001, 0.5, 2.0, 100]
etiq_lluvia = ["Sin lluvia (0 mm)", "Llovizna (0–0,5 mm)", "Lluvia (0,5–2 mm)", "Lluvia fuerte (>2 mm)"]
df["_tramo_lluvia"] = pd.cut(df[PRECIP], bins=bins_lluvia, labels=etiq_lluvia)

resumen_lluvia = df.groupby("_tramo_lluvia", observed=True)[OBJETIVO].agg(
    demanda_media="mean", n_horas="count")
prom_global = df[OBJETIVO].mean()

punta = df[df[HORA].isin([7, 8, 9, 17, 18, 19])]
res_punta = punta.groupby("_tramo_lluvia", observed=True)[OBJETIVO].mean()
prom_punta = punta[OBJETIVO].mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Demanda media según intensidad de la lluvia",
    "Mismo análisis, solo en horas punta (7–9h y 17–19h)"))

colores_ll = [color_secundario if v > prom_global else color_terciario
              for v in resumen_lluvia["demanda_media"]]
fig.add_bar(x=resumen_lluvia.index.astype(str), y=resumen_lluvia["demanda_media"],
            marker=dict(color=colores_ll, line=dict(color="white", width=1)),
            text=[f"{v:.0f} (n={int(n):,})".replace(",", ".") for v, n in
                  zip(resumen_lluvia["demanda_media"], resumen_lluvia["n_horas"])],
            textposition="outside", name="Todas las horas", row=1, col=1)
fig.add_hline(y=prom_global, line=dict(color=color_principal, dash="dash", width=1.5),
              annotation_text=f"Promedio global = {prom_global:.0f}", row=1, col=1)

colores_punta = [color_secundario if v > prom_punta else color_terciario for v in res_punta.values]
fig.add_bar(x=res_punta.index.astype(str), y=res_punta.values,
            marker=dict(color=colores_punta, line=dict(color="white", width=1)),
            text=[f"{v:.0f}" for v in res_punta.values], textposition="outside",
            name="Horas punta", row=1, col=2)
fig.add_hline(y=prom_punta, line=dict(color=color_principal, dash="dash", width=1.5),
              annotation_text=f"Promedio en punta = {prom_punta:.0f}", row=1, col=2)

fig.update_yaxes(title_text="Bicicletas por hora", row=1, col=1)
fig.update_yaxes(title_text="Bicicletas por hora", row=1, col=2)
fig.update_layout(height=480, showlegend=False)
fig.show()

caida = (1 - resumen_lluvia["demanda_media"].iloc[-1] / resumen_lluvia["demanda_media"].iloc[0]) * 100
print(f"Caída de la demanda con lluvia fuerte vs. sin lluvia: {caida:.1f}%")
print(f"Horas del dataset sin lluvia: {(df[PRECIP] == 0).mean()*100:.1f}%")


**Lectura del gráfico — y validación de la 2ª fuente.** El efecto es **grande y monótono**: a mayor
intensidad de lluvia, menor demanda, con una caída sustancial entre "sin lluvia" y "lluvia fuerte". El
segundo panel es el que hace válido el argumento: al **restringir el análisis a las horas punta**,
neutralizamos la posible confusión con el efecto de la hora (podría objetarse que llueve más de noche, cuando
la demanda ya es baja de por sí). El patrón **se mantiene**, luego el efecto es **atribuible a la lluvia** y
no a la hora.

**Conclusión sobre la decisión de arquitectura: la integración de Open-Meteo queda justificada.** El dataset
UCI original solo trae `weathersit`, una variable **categórica y gruesa** (4 niveles). Open-Meteo aporta la
**precipitación real en milímetros**, que permite distinguir una llovizna irrelevante de un aguacero que
vacía las calles. Es información que la fuente 1 **no podía entregar** y que el modelo aprovecha para afinar
precisamente los casos extremos.

**Advertencia metodológica que asumimos explícitamente:** este análisis es **descriptivo, no causal**. Las
barras "n = ..." dejan ver que las horas de lluvia fuerte son **una fracción pequeña** del total, por lo que
esa media se estima con menos datos y mayor incertidumbre.

### 8.5 Feriados: ¿justifica la 3ª fuente de datos? (P4)

Aplicamos el mismo escrutinio a la tercera fuente. Un feriado cae en día hábil pero **se comporta** como
fin de semana; si el modelo no lo sabe, esos días serán errores sistemáticos e inexplicables para él.

In [ ]:
# Comparación del PERFIL HORARIO completo, no solo del promedio.
# Razón: el promedio diario de un feriado puede parecerse al de un día normal y aun así tener
# una FORMA completamente distinta — y la forma es lo que necesita saber el operador.
lab_normal = df[df[LABORAL] == 1] if LABORAL else df[df["_es_finde"] == 0]
if FERIADO:
    lab_normal = lab_normal[lab_normal[FERIADO] == 0]
    feriados = df[df[FERIADO] == 1]
    finde = df[df["_es_finde"] == 1]

    perfiles = {
        "Día laboral normal": (lab_normal.groupby(HORA)[OBJETIVO].mean(), color_principal, "solid"),
        "Feriado":            (feriados.groupby(HORA)[OBJETIVO].mean(), color_secundario, "solid"),
        "Fin de semana":      (finde.groupby(HORA)[OBJETIVO].mean(), color_terciario, "dash"),
    }

    fig = make_subplots(rows=1, cols=2, column_widths=[0.63, 0.37], subplot_titles=(
        "Perfil horario: día laboral vs. feriado vs. fin de semana",
        "Demanda media por tipo de día"))

    for nombre, (serie_p, c, ls) in perfiles.items():
        if len(serie_p):
            fig.add_scatter(x=serie_p.index, y=serie_p.values, mode="lines+markers",
                             line=dict(color=c, dash=ls, width=2.6), marker=dict(size=5),
                             name=nombre, row=1, col=1)

    medias = pd.Series({n: s.mean() for n, (s, _, _) in perfiles.items() if len(s)})
    n_obs = pd.Series({"Día laboral normal": len(lab_normal), "Feriado": len(feriados),
                       "Fin de semana": len(finde)})
    fig.add_bar(x=medias.index, y=medias.values,
                marker=dict(color=[color_principal, color_secundario, color_terciario],
                            line=dict(color="white", width=1)),
                text=[f"{v:.0f} ({n_obs[n]:,} h)".replace(",", ".") for n, v in medias.items()],
                textposition="outside", showlegend=False, row=1, col=2)

    fig.update_xaxes(title_text="Hora del día", tickmode="linear", tick0=0, dtick=2, row=1, col=1)
    fig.update_yaxes(title_text="Demanda media", row=1, col=1)
    fig.update_yaxes(title_text="Bicicletas por hora", row=1, col=2)
    fig.update_layout(height=480)
    fig.show()

    print(f"Horas marcadas como feriado: {len(feriados):,} "
          f"({len(feriados)/len(df)*100:.1f}% del dataset)".replace(",", "."))


**Lectura del gráfico — y validación de la 3ª fuente.** La comparación de **promedios** (panel derecho) es
casi decepcionante: la diferencia de demanda media entre un feriado y un día laboral normal es **modesta**.
Si nos hubiésemos quedado ahí, habríamos concluido que la fuente 3 no aporta.

El panel izquierdo revela **lo que el promedio escondía**: el feriado **pierde el doble peak** y adopta una
forma **de meseta al mediodía**, prácticamente **superponible a la del fin de semana**. Es decir, un feriado
**es** un fin de semana disfrazado de martes. El promedio diario se parece porque la caída del peak matinal
se compensa con el aumento del mediodía —dos errores grandes de signo opuesto que se cancelan al promediar—.

**Conclusión: la fuente 3 se justifica, pero por una razón distinta de la que parecía.** Su valor no está en
mover el nivel promedio, sino en **evitar que el modelo prediga un peak de commuting que no va a ocurrir**.
Sin `is_holiday`, esas ~23 fechas al año serían errores sistemáticos: el modelo enviaría camiones de
redistribución al centro a las 8 AM de un día en que nadie va a trabajar.

**Lección metodológica (defendible en la presentación):** *comparar promedios cuando la variable tiene forma
es una trampa*. Este es el mejor ejemplo del informe de por qué una visualización bien elegida detecta lo que
un estadístico agregado oculta.

### 8.6 Estacionalidad mensual y efecto del año

Cerramos el bloque de visualizaciones cuantificando la estacionalidad que la serie temporal insinuó, ahora
separada por año para distinguir la **onda estacional** (que se repite) del **crecimiento** (que no).

In [ ]:
# Demanda media por mes, separada por año: agrupar barras permite comparar la MISMA estación
# entre años, separando visualmente estacionalidad (forma) de tendencia (nivel).
meses_lbl = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df["_mes"] = df[FECHA].dt.month
df["_anio"] = df[FECHA].dt.year

tabla_mes = df.pivot_table(index="_mes", columns="_anio", values=OBJETIVO, aggfunc="mean")
anios = list(tabla_mes.columns)

fig = go.Figure()
for i, a in enumerate(anios):
    vals = tabla_mes[a].reindex(range(1, 13)).values
    fig.add_bar(x=meses_lbl, y=vals, name=str(a), marker=dict(color=paleta[i % len(paleta)]))

prom_global = df[OBJETIVO].mean()
fig.add_hline(y=prom_global, line=dict(color=color_sexto, dash="dash", width=1.5),
              annotation_text=f"Promedio global = {prom_global:.0f}")

fig.update_layout(barmode="group", title="Demanda media por mes y año — estacionalidad y crecimiento",
                   xaxis_title="Mes", yaxis_title="Demanda media (bicicletas/hora)", height=480)
fig.show()

mejor, peor = tabla_mes.mean(axis=1).idxmax(), tabla_mes.mean(axis=1).idxmin()
print(f"Mes de mayor demanda media: {meses_lbl[mejor-1]} ({tabla_mes.mean(axis=1).max():.0f} bicis/h)")
print(f"Mes de menor demanda media: {meses_lbl[peor-1]} ({tabla_mes.mean(axis=1).min():.0f} bicis/h)")
print(f"Razón entre el mes más alto y el más bajo: "
      f"{tabla_mes.mean(axis=1).max() / tabla_mes.mean(axis=1).min():.1f}×")


**Lectura del gráfico.** Se distinguen con claridad las **dos componentes**, y esa separación es justamente
lo que el gráfico agrupado permite hacer:

- **La forma se repite** entre ambos años: valle en enero–febrero, ascenso en primavera, meseta alta en los
  meses templados, descenso hacia diciembre. Esa **onda estacional es estable** → es aprendible y por eso el
  **mes** entra como feature.
- **El nivel sube**: prácticamente cada mes del segundo año supera a su equivalente del primero. Eso es
  **tendencia**, no estacionalidad.

La distinción no es académica: la estacionalidad **se repetirá** el próximo año y el modelo puede
extrapolarla; la tendencia **puede no repetirse** (un sistema maduro deja de crecer). Un modelo de árboles
**no extrapola fuera del rango visto**, así que si la demanda sigue creciendo, **subestimará** sistemáticamente.
Esta es la fundamentación técnica del **reentrenamiento programado** que proponemos como trabajo futuro: no
es una buena intención genérica, es la consecuencia directa de este gráfico.

## 9. Análisis de correlación

Cuantificamos las relaciones que las visualizaciones mostraron. Calculamos **Pearson** (relación lineal) y lo
contrastamos con **Spearman** (relación monótona), porque ya sabemos que varias relaciones son curvas y un
solo coeficiente contaría media historia. El análisis busca tres cosas: (1) **señales predictivas**,
(2) **multicolinealidad** entre variables meteorológicas y (3) **posibles fugas de información**.

In [ ]:
# Matriz de correlación de las variables analíticas (excluimos las derivadas por legibilidad).
# El valor exacto de cada celda queda disponible en el tooltip interactivo al pasar el mouse.
cols_corr = [c for c in [OBJETIVO, HORA, TEMP, col("apparent_temperature"), HUMEDAD, PRECIP,
                         VIENTO, LABORAL, FERIADO, MES] if c is not None]
matriz = df[cols_corr].corr()

# Máscara triangular superior: la matriz es simétrica, mostrar ambas mitades duplica información.
mask = np.triu(np.ones_like(matriz, dtype=bool))
matriz_vis = matriz.mask(mask)

fig = go.Figure(data=go.Heatmap(
    z=matriz_vis.values, x=cols_corr, y=cols_corr,
    colorscale="RdBu_r", zmin=-1, zmax=1,
    colorbar=dict(title="Coeficiente de<br>correlación"),
    hovertemplate="%{y} × %{x}<br>r = %{z:.2f}<extra></extra>",
))
fig.update_layout(title="Matriz de correlación de Pearson — variables numéricas",
                   xaxis=dict(tickangle=45), yaxis=dict(autorange="reversed"), height=650)
fig.show()


In [ ]:
# Pearson vs. Spearman con el objetivo: la BRECHA entre ambos delata relaciones no lineales.
comparacion = pd.DataFrame({
    "Pearson (lineal)":   df[cols_corr].corr()[OBJETIVO],
    "Spearman (monótona)": df[cols_corr].corr(method="spearman")[OBJETIVO],
}).drop(index=OBJETIVO)
comparacion["Brecha |Spearman| - |Pearson|"] = (comparacion["Spearman (monótona)"].abs()
                                                - comparacion["Pearson (lineal)"].abs())
comparacion = comparacion.sort_values("Pearson (lineal)", key=abs, ascending=False)

print("Correlación de cada variable con la demanda (cnt):")
display(comparacion.round(3))

# Gráfico comparativo: barras agrupadas Pearson vs Spearman
fig = go.Figure()
fig.add_bar(y=comparacion.index, x=comparacion["Pearson (lineal)"], orientation="h",
            name="Pearson", marker=dict(color=color_principal))
fig.add_bar(y=comparacion.index, x=comparacion["Spearman (monótona)"], orientation="h",
            name="Spearman", marker=dict(color=color_secundario))
fig.add_vline(x=0, line=dict(color=color_quinto, width=1))

fig.update_layout(barmode="group",
                   title="Correlación con la demanda: lineal (Pearson) vs. monótona (Spearman)",
                   xaxis_title="Coeficiente de correlación con cnt",
                   yaxis=dict(autorange="reversed"), height=480)
fig.show()


**Lectura del análisis de correlación:**

- **La temperatura es la variable exógena más correlacionada** con la demanda, con signo positivo, y su
  **Spearman supera a su Pearson**: la relación es monótona pero curva, tal como mostró la Sección 8.4. Un
  modelo lineal **desaprovecharía** parte de esa señal.
- **La hora tiene una correlación lineal engañosamente baja.** Es el ejemplo pedagógico perfecto del informe:
  sabemos —por la Sección 8.2— que es un predictor **potentísimo**, pero como la relación **sube y baja dos
  veces**, el coeficiente lineal casi se cancela. **Correlación baja ≠ variable inútil**: significa
  "relación no lineal". Descartar la hora por su Pearson habría sido un error grave.
- **La humedad correlaciona negativamente** y la **precipitación** también, coherente con la lectura del
  clima: días húmedos y lluviosos desincentivan el uso.
- **Multicolinealidad entre variables meteorológicas:** `temperature_2m` y `apparent_temperature`
  (sensación térmica) están **fuertemente correlacionadas entre sí** —miden esencialmente lo mismo—. Para un
  modelo lineal esto sería un problema (coeficientes inestables); para **XGBoost** no compromete la
  predicción, pero **sí diluye la importancia de variables**: el modelo reparte el crédito arbitrariamente
  entre variables redundantes. Lo documentamos como riesgo de **interpretación**, no de rendimiento.
- **Sin *data leakage* en las variables exógenas:** ninguna variable meteorológica o de calendario alcanza
  correlaciones sospechosamente altas con el objetivo. Todas ellas son **conocidas de antemano** (el
  calendario siempre; el clima vía pronóstico), por lo que su uso en producción es legítimo. El punto crítico
  de fuga está en otra parte: en las variables autorregresivas, que auditamos a continuación.

## 10. Ingeniería de características: fundamentación desde el EDA

Todo lo anterior converge aquí. Esta sección no crea features desde cero (eso lo hace el módulo
`bikeshare.etl.features`), sino que **audita y fundamenta** las que el pipeline construyó, usando la
evidencia recogida. Es la sección más importante para la defensa: conecta **hallazgo → decisión**.

| Feature | Hallazgo del EDA que la motiva | Problema que resuelve |
|---------|-------------------------------|----------------------|
| `hour_sin`, `hour_cos` | Perfil horario cíclico (8.2) | La hora 23 y la 0 están **adyacentes**, pero como enteros distan 23 unidades |
| `month_sin`, `month_cos` | Onda estacional anual (8.6) | Mismo problema: diciembre y enero son vecinos |
| `is_weekend` / `workingday` | Dos regímenes distintos (8.3) | Permite al modelo **cambiar de patrón**, no promediarlos |
| `is_holiday` | El feriado pierde el doble peak (8.5) | Evita predecir un peak de *commuting* inexistente |
| `cnt_lag_1`, `cnt_lag_24` | Persistencia de la demanda (10.2) | Aporta el **estado actual del sistema**, no solo su contexto |
| Medias móviles | Ruido de alta frecuencia (8.1) | Resumen del nivel reciente, más estable que un solo lag |

### 10.1 Codificación cíclica: por qué `hour` no puede entrar como entero

El argumento es geométrico y se ve mejor de lo que se explica. Al proyectar la hora sobre una circunferencia
mediante $\sin(2\pi h/24)$ y $\cos(2\pi h/24)$, las 23:00 y las 00:00 quedan **contiguas**, que es su
relación real en el tiempo.

In [ ]:
# Demostración visual de la codificación cíclica.
# Elegimos este gráfico porque el problema (la discontinuidad 23→0) es GEOMÉTRICO: verlo en el
# círculo lo hace evidente de inmediato, mientras una explicación algebraica exige confiar en la fórmula.
horas = np.arange(24)
h_sin, h_cos = np.sin(2 * np.pi * horas / 24), np.cos(2 * np.pi * horas / 24)
demanda_hora = df.groupby(HORA)[OBJETIVO].mean().reindex(horas)
etiquetas_hora = [f"{h}:00" for h in horas]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Problema: la hora como entero (0–23)",
    "Solución: codificación cíclica (sin, cos) — color = demanda media"))

# --- Panel 1: la hora como entero (representación ingenua) ---
fig.add_trace(go.Scatter(x=horas, y=horas, mode="lines",
                          line=dict(color=color_quinto, width=1, dash="dot"), showlegend=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=horas, y=horas, mode="markers", text=etiquetas_hora, showlegend=False,
                          marker=dict(size=16, color=demanda_hora, colorscale=escala_bici,
                                      line=dict(color=color_principal, width=1)),
                          hovertemplate="Hora %{text}<br>Demanda media = %{marker.color:.0f}<extra></extra>"),
              row=1, col=1)
fig.add_annotation(x=0, y=0, ax=23, ay=23, xref="x", yref="y", axref="x", ayref="y",
                    showarrow=True, arrowhead=3, arrowcolor=color_sexto, arrowwidth=2, text="",
                    row=1, col=1)
fig.add_annotation(x=11, y=18, text="El modelo cree que las 23:00<br>y las 00:00 distan 23 unidades<br>(¡son vecinas!)",
                    showarrow=False, font=dict(color=color_sexto, size=11), row=1, col=1)

# --- Panel 2: la hora proyectada en el círculo unitario ---
theta = np.linspace(0, 2 * np.pi, 200)
fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines",
                          line=dict(color=color_quinto, width=1, dash="dot"), showlegend=False),
              row=1, col=2)
fig.add_trace(go.Scatter(x=h_sin, y=h_cos, mode="markers+text", text=[str(h) for h in horas],
                          textfont=dict(size=9, color="white"), showlegend=False,
                          marker=dict(size=24, color=demanda_hora, colorscale=escala_bici,
                                      colorbar=dict(title="Demanda media", x=1.02),
                                      line=dict(color=color_principal, width=1)),
                          hovertemplate="Hora %{text}<br>Demanda media = %{marker.color:.0f}<extra></extra>"),
              row=1, col=2)

fig.update_xaxes(title_text="Hora", row=1, col=1)
fig.update_yaxes(title_text="Valor que ve el modelo", row=1, col=1)
fig.update_xaxes(title_text="hour_sin", range=[-1.35, 1.35], row=1, col=2)
fig.update_yaxes(title_text="hour_cos", range=[-1.35, 1.35], scaleanchor="x2", row=1, col=2)
fig.update_layout(height=520)
fig.show()


**Lectura del gráfico.** En el panel izquierdo, la hora entera introduce una **discontinuidad artificial**:
el salto de 23 a 0 es, para el modelo, el mayor de todo el día, cuando en realidad son **60 minutos**. En el
panel derecho, la proyección circular deja las 23:00 y las 00:00 **una al lado de la otra**, y el color
revela algo adicional: **las horas de demanda similar quedan agrupadas en la misma región del círculo** —el
arco frío de la madrugada abajo, los peaks cálidos separados a los costados—. La codificación no solo repara
la discontinuidad: **ordena el espacio de features de forma coherente con el fenómeno**.

**Matiz honesto que conviene tener listo para la defensa.** Para un modelo de **árboles**, la codificación
cíclica es **menos crítica** de lo que suele afirmarse: un árbol puede aislar la hora 23 y la hora 0 en hojas
distintas mediante cortes sucesivos, sin necesidad de que estén "cerca". Su beneficio real es (1) permitir el
mismo conjunto de features si se compara con **modelos lineales o redes neuronales** —donde sí es
imprescindible—, y (2) reducir la profundidad necesaria para representar el ciclo. La mantenemos por
**comparabilidad entre modelos y buena práctica**, no porque XGBoost la exija: reconocer esto es parte del
análisis crítico.

### 10.2 Persistencia de la demanda: la evidencia detrás de los *lags* (P5)

La pregunta P5 es la de mayor impacto sobre el rendimiento del modelo: **¿la demanda de la hora anterior
predice la de esta hora?** La respondemos con un **correlograma** (autocorrelación por *lag*), que cuantifica
cuánto se parece la serie a sí misma desplazada *k* horas.

In [ ]:
# Autocorrelación de la demanda hasta 72 horas (3 días).
# Es LA herramienta para medir persistencia en series de tiempo: cada barra responde
# "¿cuánto se parece la demanda de ahora a la de hace k horas?".
serie_h = df.set_index(FECHA)[OBJETIVO].asfreq("h")   # asfreq expone los huecos reales del calendario
max_lag = 72
lags = list(range(1, max_lag + 1))
acf = [serie_h.autocorr(lag=k) for k in lags]

colores_acf = [color_secundario if k in (1, 24, 48, 72) else color_quinto for k in lags]

fig = go.Figure()
fig.add_bar(x=lags, y=acf, marker=dict(color=colores_acf, line=dict(color="white", width=0.5)),
            hovertemplate="lag %{x}h<br>r = %{y:.3f}<extra></extra>")
fig.add_hline(y=0, line=dict(color=color_principal, width=1))

# Banda de significancia aproximada (±1.96/√n): bajo ella, la correlación no se distingue de cero
lim = 1.96 / np.sqrt(len(serie_h.dropna()))
fig.add_hline(y=lim, line=dict(color=color_sexto, dash="dash", width=1),
              annotation_text="Umbral de significancia (95%)")
fig.add_hline(y=-lim, line=dict(color=color_sexto, dash="dash", width=1))

for k in (1, 24, 48):
    fig.add_annotation(x=k, y=acf[k - 1], text=f"lag {k}h<br>r = {acf[k-1]:.2f}",
                        showarrow=True, arrowhead=2, ax=35, ay=-30,
                        font=dict(color=color_secundario, size=11))

fig.update_layout(title="Autocorrelación de la demanda horaria (hasta 72 h)",
                   xaxis_title="Desfase (lag) en horas", yaxis_title="Coeficiente de autocorrelación",
                   xaxis=dict(tickmode="linear", tick0=0, dtick=6), height=460)
fig.show()

print(f"Autocorrelación lag  1 h (hora anterior)  : {acf[0]:.3f}")
print(f"Autocorrelación lag 24 h (mismo día ayer) : {acf[23]:.3f}")
print(f"Autocorrelación lag 48 h (hace dos días)  : {acf[47]:.3f}")
print(f"Autocorrelación lag 12 h (opuesta del día): {acf[11]:.3f}")


**Lectura del gráfico.** Es, probablemente, **el gráfico más importante de todo el informe**:

- **El *lag* de 1 hora tiene una autocorrelación altísima**: la demanda es **fuertemente persistente**. Si la
  última hora hubo mucho movimiento, la siguiente también lo habrá. Es intuitivo —los flujos urbanos no se
  apagan de golpe— y es una señal predictiva de primer orden.
- **Aparecen picos nítidos en los *lags* 24, 48 y 72**: la serie **se parece a sí misma cada 24 horas**. El
  correlograma "redescubre", de forma puramente estadística, el ciclo diario que ya habíamos visto en el
  perfil horario. Dos métodos independientes apuntando al mismo hallazgo es la mejor validación posible.
- **Hay un valle alrededor del *lag* 12**: la demanda de ahora es **opuesta** a la de hace 12 horas
  (mediodía vs. medianoche), lo que confirma la estructura del ciclo.

**Decisión que fundamenta:** incorporar `cnt_lag_1` (estado inmediato) y `cnt_lag_24` (mismo momento del día
anterior) como features. Esto explica, además, el resultado del modelo: en la importancia de variables **los
*lags* dominan**, y no es un accidente —**este gráfico lo anticipaba antes de entrenar nada**—.

**Reflexión crítica (obligada, y conviene anticiparla en la defensa).** Un R² de 0,96 con *lags* debe
mirarse con escepticismo sano: buena parte de ese ajuste proviene de la **inercia** de la serie, no de un
"entendimiento" de la movilidad. Un modelo ingenuo que prediga `cnt(t) = cnt(t-1)` ya alcanzaría un R² muy
respetable. **La comparación honesta es contra ese *baseline* persistente**, no contra la media. El aporte
real del clima, el calendario y el modelo se mide como la mejora **sobre** ese piso, y por eso el notebook de
modelamiento debe reportar esa comparación explícitamente.

In [ ]:
# Validación visual del lag y BASELINE ingenuo: ¿cuánto explica por sí sola la inercia?
# Este cálculo es la vara de comparación honesta para el modelo (ver reflexión anterior).
tmp = df[[FECHA, OBJETIVO]].copy().set_index(FECHA).asfreq("h")
tmp["lag_1"] = tmp[OBJETIVO].shift(1)
tmp = tmp.dropna()

# Métricas del baseline persistente: predecir "lo mismo que la hora anterior"
error = tmp[OBJETIVO] - tmp["lag_1"]
mae_base = error.abs().mean()
rmse_base = np.sqrt((error ** 2).mean())
r2_base = 1 - (error ** 2).sum() / ((tmp[OBJETIVO] - tmp[OBJETIVO].mean()) ** 2).sum()

m = tmp.sample(min(4000, len(tmp)), random_state=42)
lim_max = tmp[OBJETIVO].max()
hist_vals, _ = np.histogram(error, bins=60)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Persistencia: demanda actual vs. demanda de la hora anterior",
    "Error del baseline persistente: cnt(t) − cnt(t−1)"))

fig.add_scatter(x=m["lag_1"], y=m[OBJETIVO], mode="markers",
                 marker=dict(size=5, color=color_principal, opacity=0.35), name="Horas (muestra)",
                 row=1, col=1)
fig.add_scatter(x=[0, lim_max], y=[0, lim_max], mode="lines",
                 line=dict(color=color_secundario, dash="dash", width=2),
                 name="Identidad: cnt(t) = cnt(t-1)", row=1, col=1)

fig.add_histogram(x=error, nbinsx=60, marker=dict(color=color_septimo, line=dict(color="white", width=0.5)),
                   name="Error", row=1, col=2)
fig.add_vline(x=0, line=dict(color=color_principal, width=1.5), row=1, col=2)
fig.add_annotation(x=error.quantile(0.03), y=hist_vals.max() * 0.92,
                    text=f"MAE  = {mae_base:.1f}<br>RMSE = {rmse_base:.1f}<br>R²   = {r2_base:.3f}",
                    showarrow=False, align="left", font=dict(color=color_principal, size=12),
                    bgcolor="white", bordercolor=color_quinto, borderwidth=1, row=1, col=2)

fig.update_xaxes(title_text="cnt(t-1) — hora anterior", row=1, col=1)
fig.update_yaxes(title_text="cnt(t) — hora actual", row=1, col=1)
fig.update_xaxes(title_text="Error (bicicletas)", row=1, col=2)
fig.update_yaxes(title_text="N° de horas", tickformat=",", row=1, col=2)
fig.update_layout(height=480, showlegend=True)
fig.show()

print("BASELINE PERSISTENTE — vara de comparación mínima para el modelo:")
print(f"  MAE  = {mae_base:.1f} bicicletas/hora")
print(f"  RMSE = {rmse_base:.1f}")
print(f"  R²   = {r2_base:.3f}")
print("\nCualquier modelo debe superar CLARAMENTE estas cifras para justificar su complejidad.")


**Lectura del gráfico.** La nube se alinea sobre la diagonal de identidad, evidencia gráfica de la
persistencia. Pero se abre en abanico en los valores altos: **en las horas punta la inercia sola falla**, y
ahí es donde el modelo debe agregar valor con el clima y el calendario.

El panel derecho entrega la cifra que da **contexto honesto a las métricas del proyecto**: el baseline
persistente ya consigue un R² notable **sin aprender nada**. Es la comparación que debe acompañar al R² de
0,96 del XGBoost —un R² alto en series de tiempo persistentes es el punto de partida, no el logro—. El mérito
del modelo se mide por **cuánto reduce el MAE respecto de este piso**, especialmente en horas punta.

**Alerta de *data leakage* que este análisis obliga a verificar.** Los *lags* son legítimos **solo si** se
construyen con `shift()` **hacia el pasado y después de ordenar cronológicamente**. Dos errores clásicos los
convertirían en fuga: (1) calcularlos **antes** del `sort_values(datetime)`, y (2) usar una media móvil
**centrada** (`center=True`), que incorpora horas **futuras**. La verificamos explícitamente a continuación.

In [ ]:
# AUDITORÍA ANTI-LEAKAGE de las features autorregresivas.
# Verificamos que cada columna de lag reproduzca exactamente un shift HACIA EL PASADO del objetivo.
# Es la revisión de integridad más importante del proyecto: un lag mal construido inflaría las
# métricas de forma irreparable y el error solo aparecería en producción.
if COLS_LAG:
    filas = []
    for c in COLS_LAG:
        m = re.search(r"(\d+)", c)
        k = int(m.group(1)) if m else None
        veredicto, coincidencia = "No verificable", np.nan
        if k:
            esperado = df[OBJETIVO].shift(k)
            comparable = df[c].notna() & esperado.notna()
            coincidencia = float(np.isclose(df.loc[comparable, c], esperado[comparable]).mean() * 100)
            veredicto = "✔ Correcto (solo pasado)" if coincidencia > 99.0 else "⚠ Revisar construcción"
        filas.append({"Feature": c, "Lag (h)": k,
                      "% coincidencia con shift(k) del objetivo": round(coincidencia, 2),
                      "Veredicto": veredicto})
    auditoria = pd.DataFrame(filas).set_index("Feature")
    print("Auditoría anti-leakage de las features autorregresivas:")
    display(auditoria)
else:
    print("[aviso] No se detectaron columnas de lag en el dataset procesado.")

# Correlación de cada feature derivada con el objetivo: contexto sobre su poder predictivo
derivadas = COLS_LAG + COLS_ROLL
if derivadas:
    corr_deriv = df[derivadas + [OBJETIVO]].corr()[OBJETIVO].drop(OBJETIVO).sort_values(ascending=False)

    fig = go.Figure()
    fig.add_bar(y=corr_deriv.index, x=corr_deriv.values, orientation="h",
                marker=dict(color=color_septimo, line=dict(color="white", width=1)),
                text=[f"{v:.3f}" for v in corr_deriv.values], textposition="outside")
    fig.update_layout(title="Correlación de las features autorregresivas con la demanda",
                       xaxis_title="Correlación de Pearson con cnt", xaxis_range=[0, 1.05],
                       yaxis=dict(autorange="reversed"),
                       height=max(320, 55 * len(corr_deriv) + 150))
    fig.show()


**Lectura de la auditoría.** Cada *lag* coincide con un desplazamiento **hacia el pasado** del objetivo, lo
que confirma que **ninguna feature usa información del futuro**. Es la verificación que valida todas las
métricas reportadas: sin ella, un R² de 0,96 no sería un logro sino un síntoma.

El gráfico de correlaciones muestra por qué estas features dominan la importancia del modelo: alcanzan
coeficientes muy superiores a los de cualquier variable meteorológica. Ahora bien —y enlazando con la
reflexión anterior—, esa fuerza es a la vez su **límite**: un modelo apoyado en *lags* funciona
perfectamente para **predecir la próxima hora**, pero se degrada rápidamente si se le pide predecir **con
72 horas de anticipación**, porque entonces `cnt_lag_1` **no está disponible** y habría que alimentarlo con
predicciones propias, acumulando error. **El horizonte de predicción de esta solución es de una hora**, y eso
debe declararse explícitamente: es coherente con el caso de uso (redistribución operativa), pero no serviría
para planificación semanal.

## 11. Implicancias del EDA para el modelamiento

Recapitulamos, en formato de trazabilidad, cómo **cada hallazgo se convirtió en una decisión** del pipeline.
Esta tabla es el puente entre este notebook y el resto del proyecto (modelo, API, dashboard).

| # | Hallazgo del EDA | Decisión adoptada | Sección |
|---|------------------|-------------------|---------|
| 1 | Tendencia + estacionalidad anual | **Split temporal 80/20 sin barajar** + `TimeSeriesSplit`. Prohibido `shuffle=True` | 8.1 |
| 2 | Relación hora–demanda no lineal y con interacciones | **XGBoost** por sobre regresión lineal | 8.2, 8.3 |
| 3 | Doble régimen laboral / fin de semana | `workingday`, `is_weekend` como features de régimen | 8.3 |
| 4 | La precipitación real reduce la demanda | Se **conserva la integración con Open-Meteo** (2ª fuente validada) | 8.4 |
| 5 | El feriado pierde el doble peak | Se **conserva `is_holiday`** (3ª fuente validada) | 8.5 |
| 6 | Demanda muy persistente (ACF alta) | `cnt_lag_1`, `cnt_lag_24` + **comparación obligatoria contra baseline persistente** | 10.2 |
| 7 | Objetivo asimétrico con cola derecha | Reportar **MAE** junto al RMSE; evaluar el error **por franja horaria** | 6.1 |
| 8 | Outliers = horas punta reales | **No se eliminan ni winsorizan** | 7 |
| 9 | El modelo de árboles no extrapola la tendencia | **Reentrenamiento programado** como trabajo futuro | 8.6 |
| 10 | *Lags* verificados libres de fuga | Métricas declaradas **confiables**, con horizonte de **1 hora** | 10.2 |

In [ ]:
# Vista final del dataset que recibirá el modelo: verificación de estado.
print("Dataset listo para la etapa de modelamiento:")
print(f"  - Filas                    : {df.shape[0]:,}".replace(",", "."))
print(f"  - Columnas                 : {df.shape[1]}")
print(f"  - Nulos totales            : {int(df.isna().sum().sum())}")
print(f"  - Rango temporal           : {df[FECHA].min():%Y-%m-%d} → {df[FECHA].max():%Y-%m-%d}")
print(f"  - Demanda media / mediana  : {df[OBJETIVO].mean():.1f} / {df[OBJETIVO].median():.0f} bicis/h")

# Simulación del corte temporal 80/20 que usará el modelo, para dejarlo documentado en el EDA
corte = int(len(df) * 0.8)
fecha_corte = df[FECHA].iloc[corte]
print(f"\nCorte temporal 80/20 (sin barajar):")
print(f"  - Entrenamiento: {df[FECHA].min():%Y-%m-%d} → {fecha_corte:%Y-%m-%d}  ({corte:,} filas)".replace(",", "."))
print(f"  - Prueba       : {fecha_corte:%Y-%m-%d} → {df[FECHA].max():%Y-%m-%d}  ({len(df)-corte:,} filas)".replace(",", "."))
print(f"  - Demanda media en train / test: {df[OBJETIVO].iloc[:corte].mean():.1f} / "
      f"{df[OBJETIVO].iloc[corte:].mean():.1f} bicis/h")

display(df.dtypes.value_counts().to_frame(name="N° de columnas"))

**Lectura del corte temporal.** El bloque de prueba tiene una demanda media **superior** a la de
entrenamiento: es la consecuencia directa de la tendencia detectada en 8.1. Lejos de ser un defecto, hace la
evaluación **más exigente y más realista** —el modelo debe funcionar en un régimen que **no vio**, igual que
en producción—. Pero también implica un sesgo conocido: al no extrapolar, el modelo tenderá a **subestimar**
en el tramo final. Declararlo es parte de una evaluación honesta.

## 12. Conclusiones del EDA

**Sobre el fenómeno (lo que aprendimos del negocio):**

1. **La demanda de bicicletas es *commuting*, no paseo.** El doble peak de 8:00 y 17:00–18:00 en días
   laborales es el patrón dominante, y desaparece por completo los fines de semana y feriados, reemplazado
   por una meseta recreativa al mediodía. **Son dos sistemas distintos operando sobre la misma flota.**
2. **El clima modula, el calendario manda.** La hora y el tipo de día definen la **forma** de la demanda; el
   clima define su **amplitud**. La lluvia actúa como un freno que reduce la demanda de forma sustancial y
   monótona en su intensidad.
3. **La demanda es persistente e inercial.** La autocorrelación con la hora previa es muy alta y reaparece
   con nitidez cada 24 horas: el sistema tiene memoria.
4. **El sistema crecía.** El segundo año supera sistemáticamente al primero: los datos capturan una fase de
   adopción, no un régimen estacionario.

**Sobre el método (lo que aprendimos haciendo el análisis):**

5. **Las tres fuentes están validadas con evidencia, no por requisito.** Open-Meteo aporta la precipitación
   en mm que el `weathersit` categórico del UCI no podía dar; `holidays` evita predecir peaks inexistentes en
   ~23 fechas al año. Ambas justificaciones se sostienen en gráficos de este informe.
6. **Un estadístico agregado puede ocultar el hallazgo.** El caso de los feriados (8.5) es el ejemplo
   canónico: promedios casi iguales, formas radicalmente distintas. Y el de la hora (Sección 9): correlación
   lineal ínfima, poder predictivo enorme. **Correlación baja no significa variable inútil.**
7. **El EDA condicionó el diseño, no lo decoró.** El split temporal, la elección de XGBoost, el set de
   features, la métrica y hasta el trabajo futuro (reentrenamiento) salen de hallazgos concretos de este
   notebook, no de preferencias.

**Limitaciones reconocidas (análisis crítico):**

- **Cobertura temporal incompleta:** faltan horas en el registro original de UCI, por lo que un `shift()`
  sobre filas asume una contigüidad que no siempre se cumple. El impacto es marginal dada la alta cobertura,
  pero es una **fuente de error conocida** y no corregida.
- **Datos antiguos y de una sola ciudad:** 2011–2012, Washington D.C. Los patrones de movilidad
  post-pandemia y de otras ciudades **pueden diferir**; el modelo no es trasladable sin re-validación.
- **Agregación a nivel de sistema:** predecimos la demanda **total**, no por estación. La decisión de
  redistribución real necesita granularidad **espacial** que este dataset no entrega.
- **Análisis descriptivo, no causal:** cuando afirmamos que "la lluvia reduce la demanda" estamos leyendo una
  asociación robusta y controlada por hora, pero **no un experimento**.
- **Dependencia de los *lags*:** el horizonte útil es de **una hora**. Predicciones a varios días exigirían
  un enfoque distinto (modelos multi-horizonte o predicción recursiva con propagación de error).

**Trabajo futuro derivado del EDA:**

- Comparar el modelo contra el **baseline persistente** (Sección 10.2) y reportar la mejora **por franja
  horaria**, no solo global.
- Incorporar **granularidad espacial** (estación / clúster de estaciones) para accionar la redistribución.
- Añadir **eventos locales** (partidos, festivales) como cuarta fuente: explicarían parte de los residuos.
- **Reentrenamiento programado** con datos frescos, dada la incapacidad de los árboles para extrapolar
  tendencia.

## 13. Referencias

Fanaee-T, H., & Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge.
*Progress in Artificial Intelligence, 2*(2–3), 113–127. https://doi.org/10.1007/s13748-013-0040-3

Dua, D., & Graff, C. (2019). *Bike Sharing Dataset*. UCI Machine Learning Repository.
https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

Open-Meteo. (2024). *Historical Weather API documentation*. https://open-meteo.com/en/docs/historical-weather-api

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. En *Proceedings of the 22nd ACM
SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785–794).
https://doi.org/10.1145/2939672.2939785

Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy.
*Nature, 585*(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Plotly Technologies Inc. (2015). *Collaborative data science*. Plotly Technologies Inc.
https://plot.ly

McKinney, W. (2010). Data structures for statistical computing in Python. En *Proceedings of the 9th Python
in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: principles and practice* (3ª ed.). OTexts.
https://otexts.com/fpp3/

Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine learning in Python.
*Journal of Machine Learning Research, 12*, 2825–2830.